# Setup

In [20]:
# setup module search path
import sys
import importlib
import torch

sys.path.append('./src')

def reload_src():
    for _, mod in list(sys.modules.items()):
        if hasattr(mod, '__file__') and mod.__file__ and '/src/' in mod.__file__:
            importlib.reload(mod)

In [21]:
LOGS_DIR = './logs'
MODELS_DIR = './models'
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print(f'{device=}')

def exp_id(postfix: str) -> str:
    return f'exp_{postfix}'

device=device(type='cpu')


# 1

In [22]:
import part_1 as p1

EPOCHS_1 = 200
HIDDEN_SIZE_1 = 32
LR_1 = 1e-3

def exp_id_1(subpart: str) -> str:
    return exp_id(f'1_{subpart}')

## 1.1

Train a 2-hidden-layer MLP on these features to predict $T_c$

In [23]:
HIDDEN_SIZES_1_1 = [HIDDEN_SIZE_1] * 2 # 2 hidden layers
ACTIVATION_1_1 = 'relu'
OPTIMIZER_1_1 = 'sgd'

In [ ]:
reload_src()

p1.run_1(
    p1.RunConfig(
        exp_id=exp_id_1('1'),
        device=device,
        output=p1.Output(logs_dir=LOGS_DIR, models_dir=MODELS_DIR),
        hyperparams=p1.Hyperparams(
            epochs=EPOCHS_1,
            lr=LR_1,
            momentum=0.0,
            hidden_sizes=HIDDEN_SIZES_1_1,
            activation=ACTIVATION_1_1,
            optimizer=OPTIMIZER_1_1,
        ),
    )
)

## 1.2

Train the **same MLP** with three optimizers: SGD, SGD with momentum, and Adam. Keep the same architecture and hyperparameters, except for the optimizer. Plot the three validation loss curves on a single graph.

In [ ]:
reload_src()

configs = [
    {'exp_id':exp_id_1('2_1') ,'optimizer': 'sgd', 'momentum': 0.5},
    {'exp_id':exp_id_1('2_2'), 'optimizer': 'adam'}
]

for config in configs:
    print(f'Running with config: {config}')
    p1.run_1(
        p1.RunConfig(
            exp_id=config.get('exp_id'),
            device=device,
            output=p1.Output(logs_dir=LOGS_DIR, models_dir=MODELS_DIR),
            hyperparams=p1.Hyperparams(
                epochs=EPOCHS_1,
                lr=LR_1,
                momentum=config.get('momentum'),
                hidden_sizes=HIDDEN_SIZES_1_1,
                activation=ACTIVATION_1_1,
                optimizer=config.get('optimizer'),
            ),
        )
    )

## 1.3

Start from a deep MLP with 5 hidden layers and compare the following four configurations:

| \# | Configuration |
|---|---|
| 1 | Sigmoid + default initialization |
| 2 | ReLU + He initialization |
| 3 | Configuration 2 + BatchNorm |
| 4 | Configuration 3 + Dropout ($p=0.3$) |

For each configuration, report the final validation MSE, the mean $\ell_2$ gradient norm at the first hidden layer (averaged over mini-batches of the last epoch), and the train-validation gap.

In [26]:
OPTIMIZER_1_3 = 'adam'
HIDDEN_SIZES_1_3 = [HIDDEN_SIZE_1] * 5 # 5 hidden layers

In [ ]:
reload_src()

configs = [
    { 'exp_id': exp_id_1('3_1'), 'activation': 'sigmoid' },
    { 'exp_id': exp_id_1('3_2'), 'activation': 'relu', 'initialization': 'he' },
]
configs.append({
    **configs[1],
    'batch_norm': True,
    'exp_id': exp_id_1('3_3'),
})
configs.append({
    **configs[2],
    'dropout': 0.5,
    'exp_id': exp_id_1('3_4'),
})

for config in configs:
    print(f'Running with config: {config}')
    p1.run_1(
        p1.RunConfig(
            exp_id=config.get('exp_id'),
            device=device,
            output=p1.Output(logs_dir=LOGS_DIR, models_dir=MODELS_DIR),
            hyperparams=p1.Hyperparams(
                epochs=EPOCHS_1,
                lr=LR_1,
                momentum=config.get('momentum'),
                hidden_sizes=HIDDEN_SIZES_1_3,
                activation=config.get('activation'),
                dropout=config.get('dropout'),
                batch_norm=config.get('batch_norm'),
                optimizer=OPTIMIZER_1_3,
                initialization=config.get('initialization')
            ),
        )
    )

# 2

In [9]:
import part_2 as p2

reload_src()

EPOCHS_2 = 100
OPTIMIZER_2 = 'adam'

def exp_id_2(subpart: str) -> str:
    return exp_id(f'2_{subpart}')

## 2.1

Implement the following pipeline in PyTorch:

```
SMILES → character embedding → LSTM → final state → linear layer → prediction
```

Use correct handling of variable-length sequences. If you use padding, your aggregations and masks must ignore the padding positions.

In [10]:
HIDDEN_SIZE_2_1 = 16
EMBEDDING_SIZE_2_1 = 8
DROPOUT_2_1 = 0.5
LR_2_1 = 1e-3

In [ ]:
reload_src()

p2.run_2(
    p2.RunConfig(
        exp_id=exp_id_2('1_1'),
        model='lstm',
        device=device,
        output=p2.Output(logs_dir=LOGS_DIR, models_dir=MODELS_DIR),
        hyperparams=p2.Hyperparams(
            epochs=EPOCHS_2,
            lr=LR_2_1,
            hidden_size=HIDDEN_SIZE_2_1,
            embedding_size=EMBEDDING_SIZE_2_1,
            optimizer=OPTIMIZER_2,
            dropout=DROPOUT_2_1,
        ),
    )
)

In [ ]:
# try with a bidirectional LSTM
reload_src()

p2.run_2(
    p2.RunConfig(
        exp_id=exp_id_2('1_2'),
        model='lstm',
        device=device,
        output=p2.Output(logs_dir=LOGS_DIR, models_dir=MODELS_DIR),
        hyperparams=p2.Hyperparams(
            epochs=EPOCHS_2,
            lr=LR_2_1,
            hidden_size=HIDDEN_SIZE_2_1,
            embedding_size=EMBEDDING_SIZE_2_1,
            optimizer=OPTIMIZER_2,
            dropout=DROPOUT_2_1,
            is_bidirectional=True,
        ),
    )
)

## 2.3

Implement a transformer encoder for predicting $T_c$ with: character-by-character embedding, sinusoidal positional encoding, 2 transformer layers, mean aggregation over non-padded positions, and a final linear layer.

In [14]:
NUM_ENCODER_LAYERS_2_3 = 2
NUM_HEADS_2_3 = 4
EMBEDDING_SIZE_2_3 = 32
LR_2_3 = 1e-4
DROPOUT_2_3 = 0.1

In [ ]:
reload_src()

p2.run_2(
    p2.RunConfig(
        exp_id=exp_id_2('3'),
        model='transformer',
        device=device,
        output=p2.Output(logs_dir=LOGS_DIR, models_dir=MODELS_DIR),
        hyperparams=p2.Hyperparams(
            epochs=EPOCHS_2,
            lr=LR_2_3,
            embedding_size=EMBEDDING_SIZE_2_3,
            optimizer=OPTIMIZER_2,
            dropout=DROPOUT_2_3,
            num_encoder_layers=NUM_ENCODER_LAYERS_2_3,
            num_heads=NUM_HEADS_2_3,
        ),
    )
)